# 5. Execution Agent

The **ExecutionAgent** generates Python code for sub-tasks that require implementation and runs it in a sandboxed subprocess.

Workflow:
1. LLM generates code based on the sub-task description
2. Code is executed in a subprocess sandbox
3. stdout, stderr, and return code are captured
4. An `ExecutionResult` is returned with success/failure status


In [ ]:
from unittest.mock import AsyncMock, MagicMock
import json

from mares.agents.execution_agent import ExecutionAgent
from mares.models.sub_task import SubTask

mock_factory = MagicMock()
mock_factory.generate = AsyncMock()
mock_factory.generate.return_value = json.dumps({
    "language": "python",
    "code": "print(sum(range(1, 101)))"
})

agent = ExecutionAgent(llm_factory=mock_factory)
subtask = SubTask(id=1, task="Write a script to sum 1 to 100")

result = await agent.run(subtask)
print(f"Success: {result.success}")
print(f"stdout: {result.stdout}")
print(f"stderr: {result.stderr}")
print(f"Language: {result.language}")
print(f"Return code: {result.return_code}")

## Execution Result Fields

The `ExecutionResult` model captures full output:

In [ ]:
from mares.models.execution_result import ExecutionResult

print("ExecutionResult fields:")
for name, field in ExecutionResult.model_fields.items():
    print(f"  {name}: {field.annotation}")

## Sandbox Safety

The Python sandbox (`PythonExecutorTool`) runs code in an isolated subprocess with:
- Configurable timeout (default 10s)
- Resource limits to prevent abuse
- No access to the host filesystem beyond a temp directory

In [ ]:
from mares.tools.python_executor_tool import PythonExecutorTool

sandbox = PythonExecutorTool(timeout=5)

# Safe execution
result = await sandbox.run(code="print('hello from sandbox')")
print(f"stdout: {result.get('stdout', '')}")

# Dangerous operation is blocked by timeout
result = await sandbox.run(code="import time; time.sleep(60)")
print(f"Timed out? success={result.get('success')}, stderr={result.get('stderr', '')[:50]}")